In [4]:
import fastf1
import pandas as pd
import numpy as np
import os

# Unify cache to project root
cache_dir = 'cache/' if os.path.exists('README.md') else '../cache/'
os.makedirs(cache_dir, exist_ok=True)
fastf1.Cache.enable_cache(cache_dir)

def best_quali_time(row):
    for q in ['Q3', 'Q2', 'Q1']:
        if pd.notna(row[q]):
            return row[q].total_seconds()
    return None

def process_race(year, race_name, round_num):
    try:
        race = fastf1.get_session(year, race_name, 'R')
        race.load()
        quali = fastf1.get_session(year, race_name, 'Q')
        quali.load()
        results = race.results[['Abbreviation', 'TeamName', 
                                 'GridPosition', 'Position', 
                                 'Points', 'Status']]
        quali_results = quali.results[['Abbreviation', 'Q1', 'Q2', 'Q3']]
        df = results.merge(quali_results, on='Abbreviation', how='left')
        df['BestQualiTime'] = df.apply(best_quali_time, axis=1)
        pole_time = df['BestQualiTime'].min()
        df['QualiGapToPole'] = df['BestQualiTime'] - pole_time
        df['Year'] = year
        df['Race'] = race_name
        df['Round'] = round_num
        # If the status doesn't say 'Finished' and doesn't contain a '+' (like '+1 Lap'), it's a DNF
        df['DNF'] = (~df['Status'].astype(str).str.contains(r'Finished|\+', regex=True)).astype(int)
        df['Podium'] = (df['Position'] <= 3).astype(int)
        print(f"✓ {year} {race_name}")
        return df
    except Exception as e:
        print(f"✗ {year} {race_name}: {e}")
        return None

print("Setup complete ✓")

Setup complete ✓


In [7]:
season_2022 = pd.read_csv('data/processed/season_2022.csv')
print(f"2022 loaded: {season_2022.shape[0]} rows")
season_2021=pd.read_csv('data/processed/season_2021.csv')
print(f"2021 loaded: {season_2021.shape[0]} rows")
season_2020=pd.read_csv('data/processed/season_2020.csv')
print(f"2020 loaded: {season_2020.shape[0]} rows")
season_2023=pd.read_csv('data/processed/season_2023.csv')
print(f"2023 loaded: {season_2023.shape[0]} rows")
season_2024=pd.read_csv('data/processed/season_2024.csv')
print(f"2024 loaded: {season_2024.shape[0]} rows")
season_2025=pd.read_csv('data/processed/season_2025.csv')
print(f"2025 loaded: {season_2023.shape[0]} rows")

2022 loaded: 440 rows
2021 loaded: 440 rows
2020 loaded: 340 rows
2023 loaded: 440 rows
2024 loaded: 479 rows
2025 loaded: 440 rows


In [32]:
season_2022 = season_2022.drop_duplicates(subset=['Abbreviation', 'Year', 'Race'])
season_2022.to_csv('data/processed/season_2022.csv', index=False)
print(f"Cleaned: {season_2022.shape[0]} rows")

Cleaned: 220 rows


In [ ]:
all_seasons = [season_2023, season_2021]

for year in [2022]:
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    season_races = []
    
    for _, event in schedule.iterrows():
        df = process_race(year, event['EventName'], event['RoundNumber'])
        if df is not None:
            season_races.append(df)
    
    season_df = pd.concat(season_races, ignore_index=True)
    season_df.to_csv(f'data/processed/season_{year}.csv', index=False)
    print(f"\n{year} complete — {season_df.shape[0]} rows")
    all_seasons.append(season_df)

full_data = pd.concat(all_seasons, ignore_index=True)
full_data.to_csv('data/processed/full_dataset.csv', index=False)
print(f"\nFull dataset: {full_data.shape[0]} rows")

req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for timing_app_data. Loading data...
_api           INFO 	Fetching timing app data...
req            INFO 	Data has been written to cache!
core           INFO 	Processing timing data...
req            INFO 	No cached data found for car_data. Loading data...
_api           INFO 	Fetching car data...
_api           INFO 	Parsing car data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for position_data. Loading data...
_api           INFO 	F

✓ 2022 Bahrain Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

In [ ]:
all_seasons = [season_2023, season_2022, season_2021]

for year in [2020]:
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    season_races = []
    
    for _, event in schedule.iterrows():
        df = process_race(year, event['EventName'], event['RoundNumber'])
        if df is not None:
            season_races.append(df)
    
    season_df = pd.concat(season_races, ignore_index=True)
    season_df.to_csv(f'data/processed/season_{year}.csv', index=False)
    print(f"\n{year} complete — {season_df.shape[0]} rows")
    all_seasons.append(season_df)

full_data = pd.concat(all_seasons, ignore_index=True)
full_data.to_csv('data/processed/full_dataset.csv', index=False)
print(f"\nFull dataset: {full_data.shape[0]} rows")

In [ ]:

full_data = full_data.sort_values(by=['year', 'RoundNumber']).reset_index(drop=True)

In [ ]:
all_seasons

[    Abbreviation         TeamName  GridPosition  Position  Points    Status  \
 0            VER  Red Bull Racing           1.0       1.0    25.0  Finished   
 1            PER  Red Bull Racing           2.0       2.0    18.0  Finished   
 2            ALO     Aston Martin           5.0       3.0    15.0  Finished   
 3            SAI          Ferrari           4.0       4.0    12.0  Finished   
 4            HAM         Mercedes           7.0       5.0    10.0  Finished   
 ..           ...              ...           ...       ...     ...       ...   
 435          SAR         Williams          20.0      16.0     0.0  Finished   
 436          ZHO       Alfa Romeo          19.0      17.0     0.0  Finished   
 437          SAI          Ferrari          16.0      18.0     0.0   Retired   
 438          BOT       Alfa Romeo          18.0      19.0     0.0    Lapped   
 439          MAG     Haas F1 Team          17.0      20.0     0.0    Lapped   
 
                          Q1          

In [3]:
import os
print(os.getcwd())

C:\Users\91917\OneDrive\Desktop\f1-predictor


In [2]:
import os
os.chdir(r'C:\Users\91917\OneDrive\Desktop\f1-predictor')
print(os.getcwd())  # confirm

C:\Users\91917\OneDrive\Desktop\f1-predictor


In [ ]:

df.to_csv('data/raw/f1_raw.csv', index=False)
print(f"Saved {len(df)} rows to data/raw/f1_raw.csv ✓")

Saved 20 rows to data/raw/f1_raw.csv ✓


In [13]:
import fastf1
import os

os.makedirs('cache/', exist_ok=True)
fastf1.Cache.enable_cache('cache/')

# Check which 2022 races you have
have = set(season_2024['Race'].unique())

# Get full 2024 schedule
schedule = fastf1.get_event_schedule(2024, include_testing=False)
all_races = set(schedule['EventName'].unique())

# Find missing
missing = all_races - have
print(f"Missing {len(missing)} races:")
for r in sorted(missing):
    print(f"  - {r}")

NameError: name 'season_2024' is not defined

In [4]:
def best_quali_time(row):
    for q in ['Q3', 'Q2', 'Q1']:
        if pd.notna(row[q]):
            return row[q].total_seconds()
    return None

def process_race(year, race_name, round_num):
    try:
        race = fastf1.get_session(year, race_name, 'R')
        race.load()
        quali = fastf1.get_session(year, race_name, 'Q')
        quali.load()
        results = race.results[['Abbreviation', 'TeamName', 
                                 'GridPosition', 'Position', 
                                 'Points', 'Status']]
        quali_results = quali.results[['Abbreviation', 'Q1', 'Q2', 'Q3']]
        df_r = results.merge(quali_results, on='Abbreviation', how='left')
        df_r['BestQualiTime'] = df_r.apply(best_quali_time, axis=1)
        pole_time = df_r['BestQualiTime'].min()
        df_r['QualiGapToPole'] = df_r['BestQualiTime'] - pole_time
        df_r['Year'] = year
        df_r['Race'] = race_name
        df_r['Round'] = round_num
        df_r['DNF'] = (~df_r['Status'].astype(str).str.contains(
                        r'Finished|\+', regex=True)).astype(int)
        df_r['Podium'] = (df_r['Position'] <= 3).astype(int)
        print(f"✓ {year} {race_name}")
        return df_r
    except Exception as e:
        print(f"✗ {year} {race_name}: {e}")
        return None

# Fetch only missing races
missing_races = []
for _, event in schedule.iterrows():
    if event['EventName'] in missing:
        df_r = process_race(2022, event['EventName'], event['RoundNumber'])
        if df_r is not None:
            missing_races.append(df_r)

# Combine with existing 2022 data
season2022_complete = pd.concat([season_2022] + missing_races, ignore_index=True)
season2022_complete = season2022_complete.drop_duplicates(subset=['Abbreviation', 'Year', 'Race'])
season2022_complete.to_csv('data/processed/season_2022.csv', index=False)
print(f"\n2022 complete: {season2022_complete.shape[0]} rows")
print(f"Races: {season2022_complete['Race'].nunique()}")

NameError: name 'missing' is not defined

In [17]:
# Reload 2022 fresh
season_2022 = pd.read_csv('data/processed/season_2022.csv')
print(f"2022 rows: {season_2022.shape[0]}")
print(f"2022 races: {season_2022['Race'].nunique()}")

# Recalculate missing
have = set(season_2022['Race'].unique())
schedule = fastf1.get_event_schedule(2022, include_testing=False)
all_races = set(schedule['EventName'].unique())
missing = all_races - have

print(f"\nStill missing: {len(missing)} races")
for r in sorted(missing):
    print(f"  - {r}")

2022 rows: 440
2022 races: 22

Still missing: 0 races


In [ ]:
for year in [2024,2025]:
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    season_races = []
    
    for _, event in schedule.iterrows():
        df = process_race(year, event['EventName'], event['RoundNumber'])
        if df is not None:
            season_races.append(df)
    
    season_df = pd.concat(season_races, ignore_index=True)
    season_df.to_csv(f'data/processed/season_{year}.csv', index=False)
    print(f"\n{year} complete — {season_df.shape[0]} rows")
    all_seasons.append(season_df)

full_data = pd.concat(all_seasons, ignore_index=True)
full_data.to_csv('data/processed/full_dataset.csv', index=False)
print(f"\nFull dataset: {full_data.shape[0]} rows")

In [12]:
all_seasons=[season_2023, season_2022, season_2021, season_2020] 
all_seasons.append(season_df)


In [7]:
import os
print(os.getcwd())
os.chdir(r'C:\Users\91917\OneDrive\Desktop\f1-predictor')
print(os.getcwd())

C:\Users\91917\OneDrive\Desktop\f1-predictor
C:\Users\91917\OneDrive\Desktop\f1-predictor


In [15]:
import pandas as pd
import os

for f in os.listdir('data/processed/'):
    print(f)

driver_skill.csv
driver_skill_features.csv
final_model.pkl
full_dataset.csv
season_2020.csv
season_2021.csv
season_2022.csv
season_2023.csv
season_2024.csv
season_2025.csv


In [16]:
s2024 = pd.read_csv('data/processed/season_2024.csv')
s2025 = pd.read_csv('data/processed/season_2025.csv')

print(f"2024: {s2024.shape[0]} rows, {s2024['Race'].nunique()} races")
print(f"2025: {s2025.shape[0]} rows, {s2025['Race'].nunique()} races")

2024: 479 rows, 24 races
2025: 339 rows, 17 races


In [17]:
for year in [2025]:
    schedule = fastf1.get_event_schedule(year, include_testing=False)
    season_races = []
    
    for _, event in schedule.iterrows():
        df = process_race(year, event['EventName'], event['RoundNumber'])
        if df is not None:
            season_races.append(df)
    
    season_df = pd.concat(season_races, ignore_index=True)
    season_df.to_csv(f'data/processed/season_{year}.csv', index=False)
    print(f"\n{year} complete — {season_df.shape[0]} rows")
    all_seasons.append(season_df)

full_data = pd.concat(all_seasons, ignore_index=True)
full_data.to_csv('data/processed/full_dataset.csv', index=False)
print(f"\nFull dataset: {full_data.shape[0]} rows")

core           INFO 	Loading data for Australian Grand Prix - Race [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
core        WARNING 	Fixed incorrect tyre stint information for driver '87'
core        WARNING 	Fixed incorrect tyre stint information for driver '30'
core        WARNING 	Fixed incorrect tyre stint information for driver '5'
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core    

✓ 2025 Australian Grand Prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['81', '4', '63', '1', '31', '12', '23', '87', '18', '55', '6', '30', '7', '5', '27', '22', '14', '16', '44', '10']
core           INFO 	Loading data for Chinese Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_status

✓ 2025 Chinese Grand Prix


req            INFO 	Using cached data for session_status_data
req            INFO 	Using cached data for lap_count
req            INFO 	Using cached data for track_status_data
req            INFO 	Using cached data for _extended_timing_data
req            INFO 	Using cached data for timing_app_data
core           INFO 	Processing timing data...
req            INFO 	Using cached data for car_data
req            INFO 	Using cached data for position_data
req            INFO 	Using cached data for weather_data
req            INFO 	Using cached data for race_control_messages
core           INFO 	Finished loading data for 20 drivers: ['1', '4', '81', '16', '63', '12', '44', '6', '23', '87', '14', '22', '10', '55', '7', '27', '30', '31', '5', '18']
core           INFO 	Loading data for Japanese Grand Prix - Qualifying [v3.8.1]
req            INFO 	Using cached data for session_info
req            INFO 	Using cached data for driver_info
req            INFO 	Using cached data for session_statu

...

✓ 2025 Las Vegas Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

✓ 2025 Qatar Grand Prix


req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for _extended_timing_data. Loading data...
_api           INFO 	Fetching timing data...
_api           INFO 	Parsing timing data...
req            INFO 	Data has been written to cache!

✓ 2025 Abu Dhabi Grand Prix

2025 complete — 479 rows

Full dataset: 3416 rows


In [16]:
df.columns = df.columns.str.lower().str.strip()

df = df.rename(columns={
    'abbreviation': 'driver',
    'teamname': 'team',
    'gridposition': 'grid_position',
    'position': 'finish_position'
})

In [19]:
import pandas as pd

# LOAD DATA
df = pd.read_csv("data/processed/season_2024.csv")

print("\n--- BASIC INFO ---")
print(df.info())

print("\n--- MISSING VALUES ---")
print(df.isnull().sum())

print("\n--- DUPLICATES ---")
print("Duplicate rows:", df.duplicated().sum())

print("\n--- UNIQUE DRIVERS ---")
print(df['Abbreviation'].nunique())

print("\n--- SAMPLE DRIVER NAMES ---")
print(df['Abbreviation'].unique()[:10])

print("\n--- CHECK NUMERIC COLUMNS ---")
for col in ['GridPosition', 'Position']:
    print(f"\nColumn: {col}")
    print("Non-numeric values:")
    print(df[~pd.to_numeric(df[col], errors='coerce').notnull()][col].unique())

print("\n--- INVALID POSITIONS ---")
print("Grid <1 or >20:", df[(df['GridPosition'] < 1) | (df['GridPosition'] > 20)].shape[0])
print("Finish <1 or >20:", df[(df['Position'] < 1) | (df['Position'] > 20)].shape[0])

# CLEAN (just in case)
df['GridPosition'] = pd.to_numeric(df['GridPosition'], errors='coerce')
df['Position'] = pd.to_numeric(df['Position'], errors='coerce')

df = df.dropna(subset=['GridPosition', 'Position'])

# SIMPLE PREDICTOR (average points)
print("\n--- DRIVER RANKING (PREDICTOR) ---")
driver_avg = df.groupby('Abbreviation')['Points'].mean().sort_values(ascending=False)

print(driver_avg.head(10))



--- BASIC INFO ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 479 entries, 0 to 478
Data columns (total 16 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Abbreviation    479 non-null    object 
 1   TeamName        479 non-null    object 
 2   GridPosition    479 non-null    float64
 3   Position        479 non-null    float64
 4   Points          479 non-null    float64
 5   Status          479 non-null    object 
 6   Q1              474 non-null    object 
 7   Q2              358 non-null    object 
 8   Q3              235 non-null    object 
 9   BestQualiTime   476 non-null    float64
 10  QualiGapToPole  476 non-null    float64
 11  Year            479 non-null    int64  
 12  Race            479 non-null    object 
 13  Round           479 non-null    int64  
 14  DNF             479 non-null    int64  
 15  Podium          479 non-null    int64  
dtypes: float64(5), int64(4), object(7)
memory usage: 60.0+ KB
No

In [20]:
df = df[(df['GridPosition'] >= 1) & (df['GridPosition'] <= 20)]